# Объяснимый ИИ в СППР: кредитный скоринг
Random Forest + SHAP + LIME. Запускайте ячейки по порядку (Runtime → Run all).

In [ ]:
!pip install -q shap lime scikit-learn pandas matplotlib

## 1. Данные

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

data = fetch_openml("credit-g", version=1, as_frame=True)
X = data.data
y = (data.target == "good").astype(int)  # 1 = хороший заёмщик
print(X.shape)
X.head()

## 2. Подготовка и разделение на обучающую и тестовую части

In [ ]:
from sklearn.model_selection import train_test_split

X = pd.get_dummies(X, drop_first=True).astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## 3. Модель: Random Forest (чёрный ящик)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print("Точность:", accuracy_score(y_test, model.predict(X_test)))

## 4. SHAP: глобальное объяснение

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

shap.summary_plot(shap_values[:, :, 1], X_test, show=False)
plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. SHAP: один клиент

In [ ]:
i = 0  # номер клиента, можно менять
shap.plots.waterfall(shap_values[i, :, 1], show=False)
plt.savefig("shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Важность признаков (столбчатый график)

In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_test, plot_type="bar", show=False)
plt.show()

## 7. LIME для того же клиента

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    X_train.values, feature_names=list(X_train.columns),
    class_names=["плохой", "хороший"], mode="classification", random_state=42)
exp = lime_explainer.explain_instance(X_test.values[i], model.predict_proba, num_features=8)
fig = exp.as_pyplot_figure()
fig.savefig("lime.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Сравнение SHAP и LIME

In [ ]:
shap_top = list(pd.Series(shap_values[i, :, 1].values, index=X_test.columns).abs().sort_values(ascending=False).head(8).index)
lime_top = [X_test.columns[k] for k, w in sorted(exp.as_map()[1], key=lambda t: -abs(t[1]))[:8]]
print("SHAP:", shap_top)
print("LIME:", lime_top)
print("Общие признаки:", sorted(set(shap_top) & set(lime_top)))

## 9. Скачать графики для сайта

In [ ]:
from google.colab import files
for f in ["shap_summary.png", "shap_waterfall.png", "lime.png"]:
    files.download(f)

## Выводы
- Модель точна, но без объяснения её решения нельзя проверить.
- SHAP и LIME показывают похожие главные признаки, но не совпадают полностью.
- Объяснение — помощь эксперту, а не окончательная истина.